In [ ]:
# ── Install PyTorch compatible with SM60 (Tesla P100) ─────────────────
# PyTorch 2.5+ dropped SM60 support. 2.4.x is the last compatible version.
import subprocess
subprocess.run([
    'pip', 'install',
    'torch==2.4.1+cu121',
    'torchvision==0.19.1+cu121',
    '--index-url', 'https://download.pytorch.org/whl/cu121',
    '-q'
], check=True)

In [ ]:
# ── CUDA diagnostic (run FIRST before any training) ───────────────────
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # surface async CUDA errors immediately

import torch
print(f'torch version : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU name      : {torch.cuda.get_device_name(0)}')
    major, minor = torch.cuda.get_device_capability(0)
    print(f'Compute cap.  : SM{major}{minor}')
    print(f'Compiled archs: {torch.cuda.get_arch_list()}')
    print(f'CUDA version  : {torch.version.cuda}')

    # Sanity check: basic CUDA ops
    try:
        a = torch.tensor([1.0, 2.0]).cuda()
        b = torch.randperm(2, device='cuda')
        print(f'CUDA sanity   : OK  ({a}, {b})')
    except Exception as e:
        print(f'CUDA sanity   : FAILED -> {e}')

In [ ]:
# ── Input structure ───────────────────────────────────────────────────
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if d not in ('train_audio','test_soundscapes','train_soundscapes')]
    level = root.replace('/kaggle/input','').count(os.sep)
    print('  '*level + os.path.basename(root) + '/')
    for f in files[:6]:
        print('  '*(level+1) + f)

In [ ]:
!pip install librosa -q

In [ ]:
import time, warnings, ast
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.models as tv_models
import librosa
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

# ── Find COMP_DIR ─────────────────────────────────────────────────────
COMP_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if d not in ('train_audio','test_soundscapes','train_soundscapes')]
    if 'train.csv' in files:
        COMP_DIR = root
        break
assert COMP_DIR, 'train.csv not found'
OUT_DIR = '/kaggle/working'
print(f'COMP_DIR: {COMP_DIR}')

# ── Audio config ──────────────────────────────────────────────────────
SR = 32000; DURATION = 5; N_FFT = 1024; HOP_LEN = 320
N_MELS = 128; FMIN = 20; FMAX = 16000; IMG_W = 160

# ── Training config ───────────────────────────────────────────────────
BATCH = 32; LR = 3e-4; EPOCHS = 20
N_FOLDS = 5; TRAIN_FOLDS = [0]; SEED = 42; NUM_WORKERS = 4
np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
sub_df   = pd.read_csv(f'{COMP_DIR}/sample_submission.csv')
species_list = [c for c in sub_df.columns if c != 'row_id']
label_to_idx = {s: i for i, s in enumerate(species_list)}
print(f'Train: {len(train_df):,} recs | {len(species_list)} species')

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────
def audio_to_melspec(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP_LEN,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return mel_norm.astype(np.float32)

def load_random_chunk(path):
    try:
        dur = librosa.get_duration(path=path)
        offset = np.random.uniform(0, max(0, dur - DURATION))
        audio, _ = librosa.load(path, sr=SR, mono=True, offset=offset, duration=DURATION)
        n = DURATION * SR
        return np.pad(audio, (0, max(0, n - len(audio))))[:n]
    except Exception:
        return np.zeros(DURATION * SR, dtype=np.float32)

def spec_augment(mel):
    mel = mel.copy()
    for _ in range(2):
        f = np.random.randint(0, 15); f0 = np.random.randint(0, N_MELS - f)
        mel[f0:f0+f, :] = 0.0
    _, T = mel.shape
    for _ in range(2):
        t = np.random.randint(0, 25); t0 = np.random.randint(0, max(1, T - t))
        mel[:, t0:t0+t] = 0.0
    return mel

def mixup_cpu(x, y, alpha=0.4):
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0))  # CPU only
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────
class BirdDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        audio = load_random_chunk(f"{COMP_DIR}/train_audio/{row['filename']}")
        mel   = audio_to_melspec(audio)
        if mel.shape[1] != IMG_W:
            mel = np.array([
                np.interp(np.linspace(0, mel.shape[1]-1, IMG_W),
                          np.arange(mel.shape[1]), mel[i])
                for i in range(mel.shape[0])], dtype=np.float32)
        if self.augment:
            mel = spec_augment(mel)
        img   = np.stack([mel, mel, mel], axis=0)
        label = np.zeros(len(species_list), dtype=np.float32)
        primary = str(row['primary_label'])
        if primary in label_to_idx:
            label[label_to_idx[primary]] = 1.0
        try:
            for s in ast.literal_eval(str(row.get('secondary_labels', '[]'))):
                if str(s) in label_to_idx:
                    label[label_to_idx[str(s)]] = 0.5
        except Exception:
            pass
        return torch.from_numpy(img), torch.from_numpy(label)

In [ ]:
# ── Model: torchvision EfficientNet-B0 ───────────────────────────────
class BirdModel(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        base = tv_models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.features = base.features
        self.pool     = base.avgpool
        n_feat        = base.classifier[1].in_features  # 1280
        self.head     = nn.Sequential(nn.Dropout(0.3), nn.Linear(n_feat, n_classes))

    def forward(self, x):
        return self.head(torch.flatten(self.pool(self.features(x)), 1))

In [ ]:
# ── Training ──────────────────────────────────────────────────────────
def train_fold(fold, tr_df, vl_df):
    print(f'\n=== Fold {fold} | train={len(tr_df)}, val={len(vl_df)} ===')
    train_loader = DataLoader(BirdDataset(tr_df, augment=True),
                              batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))
    val_loader   = DataLoader(BirdDataset(vl_df, augment=False),
                              batch_size=BATCH, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))

    model     = BirdModel(n_classes=len(species_list)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.01)
    criterion = nn.BCEWithLogitsLoss()
    scaler    = GradScaler() if DEVICE == 'cuda' else None
    best_auc, best_path = 0.0, f'{OUT_DIR}/model_fold{fold}.pt'

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        t_losses = []
        for imgs, labels in train_loader:
            imgs, labels = mixup_cpu(imgs, labels)
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            if scaler:
                with autocast():
                    loss = criterion(model(imgs), labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            else:
                loss = criterion(model(imgs), labels)
                loss.backward(); optimizer.step()
            t_losses.append(loss.item())
        scheduler.step()

        model.eval()
        all_preds, all_labels, v_losses = [], [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                if scaler:
                    with autocast():
                        logits = model(imgs)
                else:
                    logits = model(imgs)
                v_losses.append(criterion(logits, labels).item())
                all_preds.append(torch.sigmoid(logits).cpu().numpy())
                all_labels.append(labels.cpu().numpy())

        preds  = np.vstack(all_preds)
        labels = np.vstack(all_labels)
        # Binarize: primary=1.0, secondary=0.5 → both positive; 0.0 → negative
        bin_labels = (labels > 0).astype(int)
        aucs   = [roc_auc_score(bin_labels[:,i], preds[:,i])
                  for i in range(bin_labels.shape[1]) if bin_labels[:,i].sum() > 0]
        auc    = np.mean(aucs) if aucs else 0.0
        elapsed = time.time() - t0
        print(f'  Ep{epoch:02d} train={np.mean(t_losses):.4f} val={np.mean(v_losses):.4f} auc={auc:.4f} ({elapsed:.0f}s)')
        if auc > best_auc:
            best_auc = auc
            torch.save({'state': model.state_dict(), 'auc': auc, 'species': species_list}, best_path)
            print(f'    -> Saved (auc={auc:.4f})')

    print(f'Fold {fold} best AUC: {best_auc:.4f}')
    return best_auc


skf    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(train_df, train_df['primary_label']))

fold_aucs = []
for fold in TRAIN_FOLDS:
    tr_idx, vl_idx = splits[fold]
    fold_aucs.append(train_fold(fold, train_df.iloc[tr_idx], train_df.iloc[vl_idx]))

print(f'\nMean CV AUC: {np.mean(fold_aucs):.4f}')

In [ ]:
import glob
for p in glob.glob(f'{OUT_DIR}/model_fold*.pt'):
    print(f'{p}: {os.path.getsize(p)/1024/1024:.1f} MB')